# FMCG Global Demand Planning and Forecasting

## Notebook 04 – Data Cleaning

### Objective

This notebook prepares the FMCG sales dataset for exploratory data analysis and machine learning by:

- Profiling the dataset
- Assessing data quality
- Handling missing values
- Removing duplicates
- Validating numerical fields
- Standardizing categorical variables
- Exporting a clean dataset

The output of this notebook will serve as the trusted dataset for all downstream analysis.

In [32]:
import pandas as pd
import numpy as np

from sqlalchemy import create_engine
import os
from dotenv import load_dotenv

load_dotenv()

True

In [33]:
USERNAME = os.getenv("DB_USER")
PASSWORD = os.getenv("DB_PASSWORD")
HOST = os.getenv("DB_HOST")
PORT = os.getenv("DB_PORT")
DATABASE = os.getenv("DB_NAME")

DATABASE_URL = (
    f"postgresql://{USERNAME}:{PASSWORD}@{HOST}:{PORT}/{DATABASE}"
)

engine = create_engine(DATABASE_URL)

print("Connected Successfully")

Connected Successfully


In [34]:
query = """
SELECT *
FROM fmcg_sales;
"""

df = pd.read_sql(query, engine)

print("Dataset Loaded Successfully")

Dataset Loaded Successfully


In [35]:
rows, cols = df.shape

print(f"Rows    : {rows:,}")
print(f"Columns : {cols}")

Rows    : 1,100,000
Columns : 33


In [37]:
df.head()

,date,year,month,day,weekofyear,weekday,is_weekend,is_holiday,temperature,rain_mm,...,discount_pct,promo_flag,gross_sales,net_sales,stock_on_hand,stock_out_flag,lead_time_days,supplier_id,purchase_cost,margin_pct
0,2023-07-21,2023,7,21,29,4,0,0,11.58,0.13,...,0.3,1,73.43,51.40,276,0,10,S041,7.51,-0.016
1,2023-07-22,2023,7,22,29,5,1,0,12.11,1.78,...,0.0,0,104.90,104.90,331,0,9,S011,7.36,0.298
2,2023-07-23,2023,7,23,29,6,1,0,15.62,1.35,...,0.0,0,73.43,73.43,219,0,3,S031,5.42,0.483
3,2023-07-24,2023,7,24,30,0,0,0,17.73,1.07,...,0.0,0,73.43,73.43,346,0,6,S047,7.31,0.303
4,2023-07-25,2023,7,25,30,1,0,0,14.24,9.28,...,0.0,0,41.96,41.96,293,0,6,S030,5.32,0.493


In [36]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1100000 entries, 0 to 1099999
Data columns (total 33 columns):
 #   Column          Non-Null Count    Dtype  
---  ------          --------------    -----  
 0   date            1100000 non-null  object 
 1   year            1100000 non-null  int64  
 2   month           1100000 non-null  int64  
 3   day             1100000 non-null  int64  
 4   weekofyear      1100000 non-null  int64  
 5   weekday         1100000 non-null  int64  
 6   is_weekend      1100000 non-null  int64  
 7   is_holiday      1100000 non-null  int64  
 8   temperature     1100000 non-null  float64
 9   rain_mm         1100000 non-null  float64
 10  store_id        1100000 non-null  object 
 11  country         1100000 non-null  object 
 12  city            1100000 non-null  object 
 13  channel         1100000 non-null  object 
 14  latitude        1100000 non-null  float64
 15  longitude       1100000 non-null  float64
 16  sku_id          1100000 non-null  ob

In [38]:
df.tail()

,date,year,month,day,weekofyear,weekday,is_weekend,is_holiday,temperature,rain_mm,...,discount_pct,promo_flag,gross_sales,net_sales,stock_on_hand,stock_out_flag,lead_time_days,supplier_id,purchase_cost,margin_pct
1099995,2023-07-16,2023,7,16,28,6,1,0,14.71,1.11,...,0.0,0,83.92,83.92,171,0,7,S053,4.92,0.531
1099996,2023-07-17,2023,7,17,29,0,0,0,16.41,4.93,...,0.0,0,62.94,62.94,234,0,9,S027,6.23,0.406
1099997,2023-07-18,2023,7,18,29,1,0,0,13.59,4.76,...,0.1,1,104.90,94.41,306,0,5,S041,6.81,0.251
1099998,2023-07-19,2023,7,19,29,2,0,0,9.32,1.74,...,0.0,0,52.45,52.45,379,0,8,S027,6.14,0.415
1099999,2023-07-20,2023,7,20,29,3,0,0,14.57,2.16,...,0.0,0,52.45,52.45,283,0,5,S052,7.22,0.312


In [39]:
df.sample(5, random_state=42)

,date,year,month,day,weekofyear,weekday,is_weekend,is_holiday,temperature,rain_mm,...,discount_pct,promo_flag,gross_sales,net_sales,stock_on_hand,stock_out_flag,lead_time_days,supplier_id,purchase_cost,margin_pct
255968,2022-11-01,2022,11,1,44,1,0,0,5.46,4.00,...,0.0,0,645.38,645.38,226,0,9,S001,6.88,0.350
780983,2022-07-14,2022,7,14,28,3,0,0,8.54,1.30,...,0.0,0,171.79,171.79,336,0,6,S052,2.32,0.447
214662,2023-09-01,2023,9,1,35,4,0,0,4.89,1.84,...,0.0,0,876.06,876.06,200,0,9,S015,5.04,0.465
339422,2023-06-23,2023,6,23,25,4,0,0,14.66,3.65,...,0.0,0,29.92,29.92,378,0,6,S038,0.74,0.455
634690,2023-09-24,2023,9,24,38,6,1,0,10.59,1.45,...,0.0,0,84.45,84.45,186,0,6,S008,3.85,0.315


In [40]:
summary = df.describe().T

summary

,count,mean,std,min,25%,50%,75%,max
year,1100000.0,2021.999668,0.816470,2021.00000,2021.00000,2022.00000,2023.00000,2023.00000
month,1100000.0,6.525613,3.447760,1.00000,4.00000,7.00000,10.00000,12.00000
day,1100000.0,15.720444,8.796262,1.00000,8.00000,16.00000,23.00000,31.00000
weekofyear,1100000.0,26.570811,15.051255,1.00000,14.00000,27.00000,40.00000,53.00000
weekday,1100000.0,3.005479,2.000451,0.00000,1.00000,3.00000,5.00000,6.00000
is_weekend,1100000.0,0.286758,0.452248,0.00000,0.00000,0.00000,1.00000,1.00000
is_holiday,1100000.0,0.013698,0.116235,0.00000,0.00000,0.00000,0.00000,1.00000
temperature,1100000.0,12.815005,3.371587,1.80000,10.61000,12.84000,15.00000,22.83000
rain_mm,1100000.0,2.904106,2.098997,0.00000,1.22000,2.57000,4.15000,11.58000
latitude,1100000.0,46.305811,4.602927,40.41706,41.90833,45.46266,52.25287,52.52586


In [41]:
missing = pd.DataFrame({
    "Missing Values": df.isnull().sum(),
    "Missing %": (df.isnull().mean()*100).round(2)
})

missing.sort_values("Missing Values", ascending=False)

,Missing Values,Missing %
date,0,0.0
year,0,0.0
month,0,0.0
day,0,0.0
weekofyear,0,0.0
weekday,0,0.0
is_weekend,0,0.0
is_holiday,0,0.0
temperature,0,0.0
rain_mm,0,0.0


In [42]:
duplicates = df.duplicated().sum()

print(f"Duplicate Rows: {duplicates:,}")

Duplicate Rows: 0


In [13]:
df = df.drop_duplicates()

print(df.shape)

(1100000, 33)


In [43]:
dtype_report = pd.DataFrame({
    "Column": df.columns,
    "Data Type": df.dtypes.values
})

dtype_report

,Column,Data Type
0,date,object
1,year,int64
2,month,int64
3,day,int64
4,weekofyear,int64
5,weekday,int64
6,is_weekend,int64
7,is_holiday,int64
8,temperature,float64
9,rain_mm,float64


In [44]:
unique_report = pd.DataFrame({
    "Column": df.columns,
    "Unique Values": df.nunique().values
})

unique_report

,Column,Unique Values
0,date,1095
1,year,3
2,month,12
3,day,31
4,weekofyear,53
5,weekday,7
6,is_weekend,2
7,is_holiday,2
8,temperature,710
9,rain_mm,559


In [45]:
print("First Date :", df["date"].min())
print("Last Date  :", df["date"].max())
print("Unique Days:", df["date"].nunique())

First Date : 2021-01-01
Last Date  : 2023-12-31
Unique Days: 1095


In [46]:
country_summary = (
    df["country"]
    .value_counts()
    .rename_axis("Country")
    .reset_index(name="Records")
)

country_summary

,Country,Records
0,Italy,350400
1,Spain,262800
2,Germany,175200
3,Austria,87600
4,Poland,87600
5,France,87600
6,Netherlands,48800


In [47]:
city_summary = (
    df["city"]
    .value_counts()
    .rename_axis("City")
    .reset_index(name="Records")
)

city_summary.head(20)

,City,Records
0,Barcelona,175200
1,Milan,175200
2,Rome,175200
3,Berlin,175200
4,Vienna,87600
5,Madrid,87600
6,Warsaw,87600
7,Paris,87600
8,Amsterdam,48800


In [48]:
print("Unique SKUs:", df["sku_id"].nunique())

df["sku_name"].value_counts().head(20)

Unique SKUs: 102


sku_name
BrandA Soda            14235
BrandF Soap            14235
BrandC Biscuits        14235
BrandB Energy drink    14235
BrandD Chips           13140
BrandC Water           13140
BrandA Toothpaste      13140
BrandF Soda            13140
BrandE Cheese          13140
BrandB Milk            13140
BrandC Shampoo         13140
BrandA Milk            13140
BrandF Water           12045
BrandF Cleaner         12045
BrandB Toothpaste      12045
BrandD Biscuits        12045
BrandB Biscuits        12045
BrandF Cheese          12045
BrandD Nuts            12045
BrandD Softener        12045
Name: count, dtype: int64

In [49]:
channel_summary = (
    df["channel"]
    .value_counts()
    .rename_axis("Channel")
    .reset_index(name="Transactions")
)

channel_summary

,Channel,Transactions
0,Hypermarket,525600
1,Supermarket,262800
2,E-commerce,262800
3,Convenience,48800


In [50]:
before = len(df)

df = df.drop_duplicates()

after = len(df)

print(f"Rows Before: {before:,}")
print(f"Rows After : {after:,}")
print(f"Duplicates Removed: {before-after:,}")

Rows Before: 1,100,000
Rows After : 1,100,000
Duplicates Removed: 0


In [51]:
text_columns = [
    "country",
    "city",
    "channel",
    "category",
    "subcategory",
    "brand",
    "sku_name"
]

for col in text_columns:
    df[col] = (
        df[col]
        .astype(str)
        .str.strip()
        .str.title()
    )

In [52]:
numeric_columns = [
    "net_sales",
    "gross_sales",
    "units_sold",
    "list_price",
    "purchase_cost",
    "stock_on_hand",
    "lead_time_days"
]

df[numeric_columns].describe()

,net_sales,gross_sales,units_sold,list_price,purchase_cost,stock_on_hand,lead_time_days
count,1.100000e+06,1.100000e+06,1.100000e+06,1.100000e+06,1.100000e+06,1.100000e+06,1.100000e+06
mean,4.299513e+02,4.406806e+02,5.919635e+01,7.712100e+00,4.625546e+00,2.994757e+02,6.500404e+00
std,4.224992e+02,4.418005e+02,4.500722e+01,4.253023e+00,2.662604e+00,8.007292e+01,2.014065e+00
min,0.000000e+00,0.000000e+00,0.000000e+00,1.080000e+00,4.900000e-01,0.000000e+00,1.000000e+00
25%,1.307675e+02,1.324400e+02,2.500000e+01,4.200000e+00,2.400000e+00,2.450000e+02,5.000000e+00
50%,2.778600e+02,2.828800e+02,4.900000e+01,7.380000e+00,4.350000e+00,3.000000e+02,6.000000e+00
75%,5.934600e+02,6.052800e+02,8.200000e+01,1.165000e+01,6.770000e+00,3.540000e+02,8.000000e+00
max,5.144940e+03,6.593900e+03,7.040000e+02,1.480000e+01,1.110000e+01,6.980000e+02,1.700000e+01


In [53]:
negative_report = pd.DataFrame({
    "Negative Count":[
        (df["net_sales"]<0).sum(),
        (df["gross_sales"]<0).sum(),
        (df["units_sold"]<0).sum(),
        (df["list_price"]<0).sum(),
        (df["purchase_cost"]<0).sum(),
        (df["stock_on_hand"]<0).sum(),
        (df["lead_time_days"]<0).sum()
    ]
},
index=[
    "Net Sales",
    "Gross Sales",
    "Units Sold",
    "List Price",
    "Purchase Cost",
    "Stock",
    "Lead Time"
])

negative_report

,Negative Count
Net Sales,0
Gross Sales,0
Units Sold,0
List Price,0
Purchase Cost,0
Stock,0
Lead Time,0


In [54]:
quality_report = pd.DataFrame({

"Metric":[
"Rows",
"Columns",
"Missing Values",
"Duplicate Rows",
"Countries",
"Cities",
"SKUs"
],

"Result":[
len(df),
df.shape[1],
df.isnull().sum().sum(),
df.duplicated().sum(),
df["country"].nunique(),
df["city"].nunique(),
df["sku_id"].nunique()
]

})

quality_report

,Metric,Result
0,Rows,1100000
1,Columns,33
2,Missing Values,0
3,Duplicate Rows,0
4,Countries,7
5,Cities,9
6,SKUs,102


In [55]:
os.makedirs("data/processed", exist_ok=True)

output_path = "data/processed/fmcg_sales_clean.csv"

df.to_csv(output_path, index=False)

print(f"Clean dataset saved to: {output_path}")

Clean dataset saved to: data/processed/fmcg_sales_clean.csv
